In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes",
    "land","wrong_fragment","urgent","hot","num_failed_logins","logged_in",
    "num_compromised","root_shell","su_attempted","num_root","num_file_creations",
    "num_shells","num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate","srv_serror_rate",
    "rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate",
    "srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate","label","difficulty"
]

train_df = pd.read_csv("data/KDDTrain.txt", names=columns)
test_df = pd.read_csv("data/KDDTest.txt", names=columns)

train_df.shape, test_df.shape

((125973, 43), (22544, 43))

In [5]:
train_df['label'].unique()

<StringArray>
[         'normal',         'neptune',     'warezclient',         'ipsweep',
       'portsweep',        'teardrop',            'nmap',           'satan',
           'smurf',             'pod',            'back',    'guess_passwd',
       'ftp_write',        'multihop',         'rootkit', 'buffer_overflow',
            'imap',     'warezmaster',             'phf',            'land',
      'loadmodule',             'spy',            'perl']
Length: 23, dtype: str

In [6]:
def map_binary(label):
    return "normal" if label == "normal" else "attack"

train_df['binary_label'] = train_df['label'].apply(map_binary)
test_df['binary_label'] = test_df['label'].apply(map_binary)

train_df['binary_label'].value_counts()

binary_label
normal    67343
attack    58630
Name: count, dtype: int64

In [7]:
categorical_cols = ['protocol_type', 'service', 'flag']

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    # handle unseen categories in test set safely
    test_df[col] = test_df[col].map(lambda s: s if s in le.classes_ else 'unknown')
    le.classes_ = np.append(le.classes_, 'unknown')
    test_df[col] = le.transform(test_df[col])
    encoders[col] = le

In [8]:
drop_cols = ['label', 'difficulty']
X_train = train_df.drop(columns=drop_cols + ['binary_label'])
y_train = train_df['binary_label']

X_test = test_df.drop(columns=drop_cols + ['binary_label'])
y_test = test_df['binary_label']

X_train.shape, X_test.shape

((125973, 41), (22544, 41))

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
target_encoder = LabelEncoder()
y_train_enc = target_encoder.fit_transform(y_train)
y_test_enc = target_encoder.transform(y_test)

target_encoder.classes_

array(['attack', 'normal'], dtype=object)

In [12]:
import joblib

np.save("data/X_train.npy", X_train_scaled)
np.save("data/X_test.npy", X_test_scaled)
np.save("data/y_train.npy", y_train_enc)
np.save("data/y_test.npy", y_test_enc)
joblib.dump(scaler, "data/scaler.pkl")
joblib.dump(encoders, "data/label_encoders.pkl")
joblib.dump(target_encoder, "data/target_encoder.pkl")
print("Saved preprocessed data and encoders.")

Saved preprocessed data and encoders.
